In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Tuned for single-worker 4-core cluster (your Databricks Cluster 1)
spark.conf.set("spark.sql.shuffle.partitions", "4")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionSize", "1mb")
spark.conf.set("spark.databricks.io.cache.enabled", "true")

CATALOG = "mini_project_team_a3t7.silver"

# ── 2. Read silver tables ─────────────────────────────────────────────────────
pos = spark.read.table("mini_project_team_a3t7.silver.pos_transactions")
wh  = spark.read.table("mini_project_team_a3t7.silver.warehouse_inventory")
pm  = spark.read.table("mini_project_team_a3t7.silver.product_master")

# ── 3. Prepare: pos_transactions → aggregate to grain ────────────────────────
# Grain: store_id + product_id + snapshot_date
pos_agg = (
    pos
    .withColumn("snapshot_date",
                F.to_date(F.col("event_timestamp").cast("timestamp")))
    .withColumn("quantity",     F.col("quantity").cast("int"))
    .withColumn("unit_price",   F.col("unit_price").cast("double"))
    .withColumn("total_amount", F.col("total_amount").cast("double"))
    .groupBy("store_id", "product_id", "snapshot_date")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("unit_price").alias("avg_selling_price"),
        F.countDistinct("transaction_id").alias("transaction_count"),
        F.countDistinct("customer_id").alias("unique_customers"),
        # Channel split
        F.sum(F.when(F.col("channel") == "online",  F.col("quantity")).otherwise(0))
         .alias("units_sold_online"),
        F.sum(F.when(F.col("channel") == "offline", F.col("quantity")).otherwise(0))
         .alias("units_sold_offline"),
        # Payment method split
        F.sum(F.when(F.col("payment_method") == "cash",           F.col("total_amount")).otherwise(0))
         .alias("revenue_cash"),
        F.sum(F.when(F.col("payment_method") == "credit_card",    F.col("total_amount")).otherwise(0))
         .alias("revenue_credit_card"),
        F.sum(F.when(F.col("payment_method") == "debit_card",     F.col("total_amount")).otherwise(0))
         .alias("revenue_debit_card"),
        F.sum(F.when(F.col("payment_method") == "mobile_payment", F.col("total_amount")).otherwise(0))
         .alias("revenue_mobile_payment"),
    )
)

# ── 4. Prepare: warehouse_inventory → collapse to 1 row per product ───────────
wh_agg = (
    wh
    .withColumn("current_stock_qty",   F.col("current_stock_qty").cast("int"))
    .withColumn("reserved_stock_qty",  F.col("reserved_stock_qty").cast("int"))
    .withColumn("available_stock_qty", F.col("available_stock_qty").cast("int"))
    .withColumn("reorder_level",       F.col("reorder_level").cast("int"))
    .withColumn("max_stock",           F.col("max_stock").cast("int"))
    .groupBy("product_id")
    .agg(
        F.sum("current_stock_qty").alias("current_stock_qty"),
        F.sum("reserved_stock_qty").alias("reserved_stock_qty"),
        F.sum("available_stock_qty").alias("available_stock_qty"),
        F.max("reorder_level").alias("reorder_level"),
        F.sum("max_stock").alias("max_stock"),
        F.first("location_zone").alias("primary_location_zone"),
        F.countDistinct("warehouse_id").alias("warehouse_count"),
        F.collect_set("warehouse_id").alias("warehouse_ids"),
        F.max("last_updated").alias("last_stock_updated"),
    )
)

# ── 5. Prepare: product_master → deduplicate to 1 row per product ─────────────
pm_window = Window.partitionBy("product_id").orderBy(F.col("effective_date").desc())

pm_dedup = (
    pm
    .withColumn("cost_price",    F.col("cost_price").cast("double"))
    .withColumn("selling_price", F.col("selling_price").cast("double"))
    .withColumn("weight",        F.col("weight").cast("double"))
    .withColumn("rn", F.row_number().over(pm_window))
    .filter(F.col("rn") == 1)
    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "supplier_id",
        "cost_price",
        "selling_price",
        "weight",
        "status",
        F.col("effective_date").alias("product_effective_date"),
    )
)

# ═══════════════════════════════════════════════════════════════════════════════
# GOLD TABLE 1 — fact_sales
# Scope: POS ∩ product_master (~24,509 SKUs)
# Purpose: revenue, margin, category, brand, channel analysis
# ═══════════════════════════════════════════════════════════════════════════════

fact_sales_raw = (
    pos_agg
    .join(F.broadcast(pm_dedup), on="product_id", how="inner")
)

fact_sales = (
    fact_sales_raw

    # ── Profitability ──────────────────────────────────────────────────────
    .withColumn(
        "gross_profit",
        F.round(
            (F.col("avg_selling_price") - F.col("cost_price")) * F.col("units_sold"),
            2
        )
    )
    .withColumn(
        "gross_margin_pct",
        F.round(
            F.when(
                F.col("total_revenue") > 0,
                (F.col("total_revenue") - F.col("cost_price") * F.col("units_sold"))
                / F.col("total_revenue") * 100
            ).otherwise(0.0),
            2
        )
    )
    .withColumn(
        "profit_per_unit",
        F.round(F.col("avg_selling_price") - F.col("cost_price"), 2)
    )

    # ── Channel & payment ratios ───────────────────────────────────────────
    .withColumn(
        "online_sales_pct",
        F.round(
            F.when(F.col("units_sold") > 0,
                   F.col("units_sold_online") / F.col("units_sold") * 100
            ).otherwise(0.0),
            1
        )
    )
    .withColumn(
        "digital_payment_pct",
        F.round(
            F.when(
                F.col("total_revenue") > 0,
                (F.col("revenue_credit_card") + F.col("revenue_debit_card")
                 + F.col("revenue_mobile_payment")) / F.col("total_revenue") * 100
            ).otherwise(0.0),
            1
        )
    )

    # ── Surrogate key ─────────────────────────────────────────────────────
    .withColumn(
        "fact_sales_key",
        F.sha2(
            F.concat_ws("|",
                F.col("store_id"),
                F.col("product_id"),
                F.col("snapshot_date").cast("string")
            ),
            256
        )
    )
    .withColumn("dw_created_at", F.current_timestamp())
)

fact_sales_final = fact_sales.select(
    "fact_sales_key",
    "snapshot_date",
    "store_id",
    "product_id",
    "supplier_id",
    "product_name",
    "category",
    "subcategory",
    "brand",
    "status",
    "cost_price",
    "selling_price",
    "units_sold",
    "total_revenue",
    "avg_selling_price",
    "transaction_count",
    "unique_customers",
    "units_sold_online",
    "units_sold_offline",
    "revenue_cash",
    "revenue_credit_card",
    "revenue_debit_card",
    "revenue_mobile_payment",
    "gross_profit",
    "gross_margin_pct",
    "profit_per_unit",
    "online_sales_pct",
    "digital_payment_pct",
    "dw_created_at",
)

# ═══════════════════════════════════════════════════════════════════════════════
# GOLD TABLE 2 — fact_inventory_full
# Scope: POS ∩ product_master ∩ warehouse (~872 SKUs)
# Purpose: stock health, reorder alerts, sell-through, inventory KPIs
# ═══════════════════════════════════════════════════════════════════════════════

w30_cover = (
    Window.partitionBy("store_id", "product_id")
          .orderBy("snapshot_date")
          .rowsBetween(-30, 0)
)
 
# ── Join (unchanged) ──────────────────────────────────────────────────────────
fact_inv_raw = (
    pos_agg
    .join(F.broadcast(pm_dedup), on="product_id", how="inner")
    .join(F.broadcast(wh_agg),   on="product_id", how="inner")
)
 
# ── Derived columns ───────────────────────────────────────────────────────────
fact_inv = (
    fact_inv_raw
 
    # ── Profitability (unchanged) ──────────────────────────────────────────
    .withColumn(
        "gross_profit",
        F.round(
            (F.col("avg_selling_price") - F.col("cost_price")) * F.col("units_sold"),
            2
        )
    )
    .withColumn(
        "gross_margin_pct",
        F.round(
            F.when(
                F.col("total_revenue") > 0,
                (F.col("total_revenue") - F.col("cost_price") * F.col("units_sold"))
                / F.col("total_revenue") * 100
            ).otherwise(0.0),
            2
        )
    )
    .withColumn(
        "profit_per_unit",
        F.round(F.col("avg_selling_price") - F.col("cost_price"), 2)
    )
 
    # ── FIX: Compute 30-day rolling avg daily sales BEFORE stock_cover_days ──
    .withColumn(
        "avg_daily_sales_30d",
        F.round(F.avg("units_sold").over(w30_cover), 2)
    )
 
    # ── FIX: stock_cover_days now uses avg_daily_sales_30d as denominator ─────
  
    .withColumn(
        "stock_cover_days",
        F.round(
            F.when(
                F.col("avg_daily_sales_30d") > 0,       # was: units_sold > 0
                F.col("available_stock_qty") / F.col("avg_daily_sales_30d")
            ).otherwise(F.lit(None).cast("double")),
            1
        )
    )
 
    # ── Inventory health flags (unchanged logic, correct denominator now) ──
    .withColumn(
        "reorder_flag",
        F.when(F.col("available_stock_qty") <= F.col("reorder_level"),
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "stockout_flag",
        F.when(F.col("available_stock_qty") == 0,
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "overstock_flag",
        F.when(F.col("current_stock_qty") > F.col("max_stock") * 0.9,
               F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "sell_through_rate",
        F.round(
            F.when(
                (F.col("units_sold") + F.col("available_stock_qty")) > 0,
                F.col("units_sold") / (F.col("units_sold") + F.col("available_stock_qty")) * 100
            ).otherwise(0.0),
            2
        )
    )
    .withColumn(
        "stock_utilisation_pct",
        F.round(
            F.when(F.col("max_stock") > 0,
                   F.col("current_stock_qty") / F.col("max_stock") * 100
            ).otherwise(0.0),
            1
        )
    )
    .withColumn(
        "online_sales_pct",
        F.round(
            F.when(F.col("units_sold") > 0,
                   F.col("units_sold_online") / F.col("units_sold") * 100
            ).otherwise(0.0),
            1
        )
    )
 
    # ── Surrogate key (unchanged) ──────────────────────────────────────────
    .withColumn(
        "inventory_snapshot_key",
        F.sha2(
            F.concat_ws("|",
                F.col("store_id"),
                F.col("product_id"),
                F.col("snapshot_date").cast("string")
            ),
            256
        )
    )
    .withColumn("dw_created_at", F.current_timestamp())
)
 
# ── Final select (avg_daily_sales_30d added to output) ───────────────────────
fact_inv_final = fact_inv.select(
    "inventory_snapshot_key",
    "snapshot_date",
    "store_id",
    "product_id",
    "supplier_id",
    "product_name",
    "category",
    "subcategory",
    "brand",
    "status",
    "cost_price",
    "selling_price",
    "units_sold",
    "avg_daily_sales_30d",       
    "total_revenue",
    "avg_selling_price",
    "transaction_count",
    "unique_customers",
    "units_sold_online",
    "units_sold_offline",
    "revenue_cash",
    "revenue_credit_card",
    "revenue_debit_card",
    "revenue_mobile_payment",
    "gross_profit",
    "gross_margin_pct",
    "profit_per_unit",
    "current_stock_qty",
    "reserved_stock_qty",
    "available_stock_qty",
    "reorder_level",
    "max_stock",
    "warehouse_count",
    "primary_location_zone",
    "warehouse_ids",
    "last_stock_updated",
    "stock_cover_days",          
    "sell_through_rate",
    "stock_utilisation_pct",
    "online_sales_pct",
    "reorder_flag",
    "stockout_flag",
    "overstock_flag",
    "dw_created_at",
)



In [0]:
print("Writing gold.fact_sales ...")

(
    fact_sales_final
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("snapshot_date")
    .saveAsTable("mini_project_team_a3t7.gold.fact_sales")
)


In [0]:
print("Writing gold.fact_inventory_full ...")

(
    fact_inv_final
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("snapshot_date")
    .saveAsTable("mini_project_team_a3t7.gold.fact_inventory_full")
)

print("✅ gold.fact_inventory_full done.")

In [0]:
from pyspark.sql.functions import (
    col, sum, count, avg, round, when,
    lit, greatest, coalesce, date_trunc
)
from delta.tables import DeltaTable

gold_table = "mini_project_team_a3t7.gold.daily_inventory_fact"

# ── Step 1: Read Streaming POS Data ───────────────────────

df_pos = (
    spark.readStream
    .table("mini_project_team_a3t7.silver.pos_transactions")
)

# Static dimension tables
df_wh = spark.table("mini_project_team_a3t7.silver.warehouse_inventory")
df_sinv = spark.table("mini_project_team_a3t7.silver.inventory_snapshots")
df_prod = spark.table("mini_project_team_a3t7.silver.product_master")

# ── Step 2: Daily Sales Aggregation ───────────────────────

daily_sales = (
    df_pos
    .withWatermark("event_timestamp", "1 day")
    .withColumn("inventory_date", date_trunc("day", col("event_timestamp")))
    .groupBy("store_id", "product_id", "inventory_date")
    .agg(
        sum("quantity").alias("units_sold"),
        sum("total_amount").alias("revenue"),
        count("*").alias("transaction_count"),
        avg("unit_price").alias("avg_unit_price")
    )
)

# ── Step 3: Warehouse Aggregation ─────────────────────────

wh_by_product = (
    df_wh
    .groupBy("product_id")
    .agg(
        sum("current_stock_qty").alias("current_stock"),
        sum("available_stock_qty").alias("available_stock"),
        avg("reorder_level").alias("reorder_level"),
        avg("max_stock").alias("max_stock")
    )
)

# ── Step 4: Product Attributes ───────────────────────────

prod_slim = (
    df_prod.select(
        "product_id",
        "cost_price",
        "selling_price",
        "category",
        "supplier_id"
    )
)

# ── Step 5: Store Inventory Snapshot ─────────────────────

store_inv_slim = (
    df_sinv.select(
        "store_id",
        "product_id",
        col("inventory_quantity").alias("current_quantity"),
        "expiry_date"
    )
)

# ── Step 6: Build Streaming Fact ─────────────────────────

fact_stream = (
    daily_sales
    .join(wh_by_product, "product_id", "left")
    .join(prod_slim, "product_id", "left")
    .join(store_inv_slim, ["store_id", "product_id"], "left")

    .withColumn(
        "stock_value",
        round(
            coalesce(col("available_stock"), lit(0)) *
            coalesce(col("cost_price"), lit(0)), 2
        )
    )

    .withColumn(
        "days_on_hand",
        when(
            col("units_sold") > 0,
            round(
                coalesce(col("current_quantity"), lit(0)) /
                col("units_sold"), 1
            )
        ).otherwise(lit(999))
    )

    .withColumn(
        "reorder_quantity",
        greatest(
            lit(0),
            coalesce(col("reorder_level"), lit(0)) -
            coalesce(col("available_stock"), lit(0))
        )
    )

    .withColumn(
        "is_stockout",
        coalesce(col("current_quantity"), lit(0)) == 0
    )

    .withColumn(
        "is_low_stock",
        (coalesce(col("current_quantity"), lit(0)) > 0) &
        (coalesce(col("current_quantity"), lit(0)) <= 20)
    )
)

# ── Step 7: Upsert Function (Delta Merge) ─────────────────

def upsert_fact(batch_df, batch_id):

    if spark.catalog.tableExists(gold_table):

        delta_table = DeltaTable.forName(spark, gold_table)

        (
            delta_table.alias("t")
            .merge(
                batch_df.alias("s"),
                """
                t.store_id = s.store_id
                AND t.product_id = s.product_id
                AND t.inventory_date = s.inventory_date
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    else:

        (
            batch_df.write
            .format("delta")
            .partitionBy("inventory_date")
            .saveAsTable(gold_table)
        )

# ── Step 8: Write Streaming Output ───────────────────────

query = (
    fact_stream.writeStream
    .foreachBatch(upsert_fact)
    .outputMode("update")
    .option(
        "checkpointLocation",
        "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/gold/daily_inventory_fact/"
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()